In [1]:
import os
import io
import sys
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

In [2]:
load_dotenv(override = True)
groq_api_key = os.getenv("GROQ_API_KEY") or os.getenv("OPENAI_API_KEY")


In [3]:
baseUrl = "https://api.groq.com/openai/v1"
if not groq_api_key:
    raise RuntimeError("Missing API key. Set GROQ_API_KEY or OPENAI_API_KEY in .env or environment.")
groq = OpenAI(api_key=groq_api_key, base_url=baseUrl,timeout=60.0 )


In [4]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()

In [5]:
message = f"""
Here is a report of the system information for my computer.

I want to run a single JavaScript file called `main.js` and execute it in the simplest and fastest way possible.

Please determine whether my system already has a JavaScript runtime installed (such as Node.js). If not, provide the simplest step-by-step instructions to install the recommended runtime.

If my system is already configured to run JavaScript, I would like to execute it from Python using `subprocess.run()`.

Please provide the exact values I should use for the commands below:

```python
run_command = # something here
run_result = subprocess.run(
    run_command,
    check=True,
    text=True,
    capture_output=True
)
return run_result.stdout
```

If there is a separate compile or build step that would improve runtime performance for a single JavaScript file, include it. Otherwise, explain why no compilation step is needed and provide only the appropriate `run_command`.

If multiple JavaScript runtimes are available (e.g., Node.js, Deno, Bun), recommend the simplest and most widely supported option.

System information:
{system_info}
"""

response = groq.chat.completions.create(model="llama-3.1-8b-instant",messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))


Based on your system information, you're running Windows 10. Given that Windows has limited support for native JavaScript execution, I recommend using Node.js as the JavaScript runtime environment.

**Installing Node.js**

Node.js is widely used and well-supported. To install it on your system, you can use the Windows Package Manager, `winget`. Here are the simple step-by-step instructions:

1. Open the Windows Terminal or Command Prompt as Administrator.
2. Download and install the `winget` command-line tool (if it's not already installed):
   ```bash
winget install --id Microsoft.DesktopAppInstaller
```
3. Install Node.js and npm (the package manager for Node.js):
   ```bash
winget install --id Microsoft.Nodejs --version 18.21.2
```
   Make sure to update Node.js to the latest version available from the store.
4. Verify the installation:
   ```bash
node -v
```

**Executing the JavaScript file using Node.js and Python**

If Node.js is installed correctly, you can execute the `main.js` file using Node.js from Python. Here are the values you should use in your code:

```python
import subprocess

# The command to run Node.js and execute main.js
run_command = 'node main.js'

run_result = subprocess.run(
    run_command,
    check=True,
    text=True,
    capture_output=True
)
return run_result.stdout
```

Since Node.js is an interpreted language and does not require a compilation step, you can execute the JavaScript file directly. The `node` command is the one that loads the compiled JavaScript and interprets it.

**Alternative JavaScript Runtimes (Not Recommended)**

For the sake of completeness, here's a brief look at some other JavaScript runtimes you might consider:

* **Deno**: Requires Rust to be installed and may not be the best option on Windows.
* **Bun**: Not as widely used or well-documented as Node.js and has limited support on Windows.

Considering the simplicity and Windows support, **Node.js** is the recommended choice. If you decide to use an alternative, you'll need to adjust the installation and execution steps accordingly.

In [6]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with JavaSCript code. Do not provide any explanation other than occasional comments.
The JavaScript response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to JavaScript with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.js and then compiled and executed; the compilation command is:
Respond only with JavaScript code.
Python code to port:

```python
{python}
```
"""

In [7]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [8]:
def write_output(js):
    with open("main.js", "w", encoding="utf-8") as f:
        f.write(js)

In [9]:
def port(python):
    response = groq.chat.completions.create(model="llama-3.1-8b-instant", messages=messages_for(python))
    reply = response.choices[0].message.content
    reply = reply.replace('```js','').replace('```','')
    write_output(reply)

In [10]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [11]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [12]:
port(pi)

In [15]:
import subprocess

run_command = ["node", "main.js"]

def run_javascript():
    for _ in range(3):
        result = subprocess.run(
            run_command,
            text=True,
            capture_output=True
        )

        print("Exit code:", result.returncode)
        print("STDOUT:")
        print(result.stdout)
        print("STDERR:")
        print(result.stderr)

        if result.returncode != 0:
            break

In [17]:
run_javascript()

Exit code: 0
STDOUT:
2 + 3 = 5

STDERR:

Exit code: 0
STDOUT:
2 + 3 = 5

STDERR:

Exit code: 0
STDOUT:
2 + 3 = 5

STDERR:

